# 2. 循环结构

循环是 LangGraph 最核心的能力：**边可以指回上游节点**，让 Graph 反复执行，直到条件边放行到 END。ReAct（思考 → 行动 → 观察 → 再思考）、生成-检查-重写都是循环。

两个关键点：

- 循环**必须**配合条件边：否则图会一直转下去。LangGraph 用 **recursion_limit** 兜底（默认 25 步），超过就抛 `GraphRecursionError`
- 每转一圈 State 都保留：`add_messages`、计数器 reducer 让 State 随循环累积——这是循环「越跑越接近目标」的前提

In [ ]:
# 示例：计数器循环 —— 转 3 圈自动退出（纯 Python，不依赖 LLM）

from operator import add
from typing import Annotated, TypedDict

from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from IPython.display import display

class LoopState(TypedDict):
    count: Annotated[int, add]


def inc(state: LoopState) -> dict:
    print(f"第 {state['count']} 圈")
    return {"count": 1}


def route(state: LoopState) -> str:
    return "loop" if state["count"] < 3 else END   # 满 3 圈放行到 END


builder = StateGraph(state_schema=LoopState)
builder.add_node("inc", inc)
builder.add_edge(START, "inc")
# 循环的关键：条件边的映射把 inc 指回自己；END 也放进映射表，route 返回 END 时直接结束
builder.add_conditional_edges("inc", route, {"loop": "inc", END: END})

graph = builder.compile()
display(graph)
print(graph.invoke({"count": 0}))
# 第 0 圈
# 第 1 圈
# 第 2 圈
# {'count': 3}


# recursion_limit 兜底：故意让循环永不退出，验证防死循环保护（默认上限 25 步）
def always_loop(state: LoopState) -> str:
    return "loop"


builder2 = StateGraph(state_schema=LoopState)
builder2.add_node("inc", inc)
builder2.add_edge(START, "inc")
builder2.add_conditional_edges("inc", always_loop, {"loop": "inc"})

try:
    builder2.compile().invoke({"count": 0}, config={"recursion_limit": 10})
except GraphRecursionError as e:
    print("触发了防死循环保护:", str(e)[:60])

In [ ]:
# 示例：LLM 自纠错循环 —— 写诗直到包含「月亮」，不合格就重写

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class PoemState(MessagesState):
    ok: bool


def write_poem(state: PoemState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content="写一首描述太阳或月亮的短诗")]
    )
    return {"messages": [response]}


def check(state: PoemState) -> dict:
    last = state["messages"][-1]
    ok = "月亮" in (last.content or "")
    print(f"检查: 诗中{'包含' if ok else '不含'}月亮")
    return {"ok": ok}


def route(state: PoemState) -> str:
    return "finish" if state["ok"] else "write_poem"   # 不合格 → 指回写诗节点重写


def finish(state: PoemState) -> dict:
    return {}


builder = StateGraph(state_schema=PoemState)
builder.add_node("write_poem", write_poem)
builder.add_node("check", check)
builder.add_node("finish", finish)
builder.add_edge(START, "write_poem")
builder.add_edge("write_poem", "check")
# 循环的关键：check 的条件边把 write_poem 指回自己，同时保留通往 finish 的分支
builder.add_conditional_edges("check", route, {"write_poem": "write_poem", "finish": "finish"})
builder.add_edge("finish", END)

result = builder.compile().invoke({"messages": [HumanMessage(content="开始")]})
rprint(result["messages"][-1].content)

In [ ]:
from operator import add
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    response: str
    stop: Literal["failure", "success"]


def llm_node(state: ChatState) -> dict:
    response = model.invoke(state["messages"])
    return {"response": response.content, "messages": [response]}


def check_node(state: ChatState) -> dict:
    words = ["股票", "摄影", "天气", "金融"]
    for word in words:
        if word in state["response"]:
            feedback = HumanMessage(
                content=f"上一段文字包含不允许内容「{word}」，请避开该主题重新写"
            )
            return {"stop": "failure", "messages": [feedback]}
    return {"stop": "success"}


def finish_node(state: ChatState) -> dict:
    return {}


def route_node(state: ChatState) -> str:
    return "llm_node" if state["stop"] == "failure" else "finish_node"


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("check_node", check_node)
builder.add_node("finish_node", finish_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "check_node")
builder.add_conditional_edges(
    "check_node", route_node, {"llm_node": "llm_node", "finish_node": "finish_node"}
)
builder.add_edge("finish_node", END)

graph = builder.compile()
display(graph)

result = graph.invoke(
    {
        "check_count": 0,
        "messages": [
            HumanMessage(
                content="请写一段简短文字，主题从金融、股票、摄影、天气、运动中选择一个"
            )
        ],
    }
)
rprint(result)